# BirdCLEF 2026 — CNN + Transformer Hybrid — Inference Only (Pipeline 02)

This notebook performs **inference only** using a pre-trained hybrid model checkpoint and generates `submission.csv`.

**Instructions:**
- Attach your trained model dataset containing `best_model_hybrid.pth`.
- Update `CFG.MODEL_PATH` to point to the checkpoint.
- Run all cells → `submission.csv` is written to `/kaggle/working/`.

## 1. Imports

In [ ]:
import os
import gc
import math
import glob
import random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import soundfile as sf
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio.transforms as T
import timm

import warnings
warnings.filterwarnings('ignore')

## 2. Configuration

In [ ]:
class Config:
    ROOT_DIR    = '/kaggle/input/competitions/birdclef-2026'
    TRAIN_CSV   = os.path.join(ROOT_DIR, 'train.csv')
    SOUNDSCAPE_DIR = os.path.join(ROOT_DIR, 'train_soundscapes')

    # Point this to your trained checkpoint
    MODEL_PATH  = 'best_model_hybrid.pth'  # or '/kaggle/input/your-dataset/best_model_hybrid.pth'

    # Audio
    SR             = 32000
    WINDOW_SECONDS = 5

    # Mel Spectrogram
    N_MELS     = 128
    N_FFT      = 2048
    HOP_LENGTH = 512
    FMIN       = 20
    FMAX       = 16000

    # Transformer hyper-params must match the training run
    CNN_BACKBONE = 'tf_efficientnet_b0'
    D_MODEL      = 256
    NHEAD        = 8
    NUM_LAYERS   = 4
    DIM_FF       = 1024
    DROPOUT      = 0.1
    NUM_CLASSES  = 0  # populated automatically

CFG = Config()

print('Loading label schema...')
train_df   = pd.read_csv(CFG.TRAIN_CSV)
sample_sub = pd.read_csv(os.path.join(CFG.ROOT_DIR, 'sample_submission.csv'))
train_labels      = sorted(train_df['primary_label'].unique())
submission_labels = [c for c in sample_sub.columns if c != 'row_id']
unique_labels     = submission_labels
CFG.NUM_CLASSES   = len(unique_labels)

print(f'Clip labels: {len(train_labels)} | Submission classes: {CFG.NUM_CLASSES}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


## 3. Model Definition

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=4096, dropout=0.1):
        super().__init__()
        self.dropout    = nn.Dropout(dropout)
        self.embedding  = nn.Embedding(max_len, d_model)

    def forward(self, x):
        pos = torch.arange(x.size(1), device=x.device).unsqueeze(0)
        return self.dropout(x + self.embedding(pos))


class CNNTransformerHybrid(nn.Module):
    def __init__(self, cnn_backbone, num_classes, d_model, nhead,
                 num_layers, dim_ff, dropout, model_path=None):
        super().__init__()
        self.cnn = timm.create_model(
            cnn_backbone, pretrained=False, in_chans=3, features_only=True
        )
        with torch.no_grad():
            dummy     = torch.zeros(1, 3, CFG.N_MELS, CFG.N_MELS)
            feat_ch   = self.cnn(dummy)[-1].shape[1]

        self.token_proj = nn.Sequential(
            nn.Conv2d(feat_ch, d_model, kernel_size=1, bias=False),
            nn.BatchNorm2d(d_model), nn.GELU()
        )
        self.pos_enc = PositionalEncoding(d_model, max_len=4096, dropout=dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, activation='gelu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(d_model, num_classes))

    def forward(self, x):
        f      = self.cnn(x)[-1]
        f      = self.token_proj(f)
        B,D,H,W = f.shape
        tokens = f.flatten(2).transpose(1, 2)
        tokens = self.pos_enc(tokens)
        tokens = self.transformer(tokens)
        pooled = tokens.mean(dim=1)
        return self.head(pooled)


## 4. Load Checkpoint

In [ ]:
print('Initialising model...')
model = CNNTransformerHybrid(
    cnn_backbone=CFG.CNN_BACKBONE,
    num_classes=CFG.NUM_CLASSES,
    d_model=CFG.D_MODEL, nhead=CFG.NHEAD,
    num_layers=CFG.NUM_LAYERS, dim_ff=CFG.DIM_FF,
    dropout=CFG.DROPOUT, model_path=None
).to(device)

try:
    model.load_state_dict(torch.load(CFG.MODEL_PATH, map_location=device))
    model.eval()
    print(f'✅ Checkpoint loaded from {CFG.MODEL_PATH}')
except Exception as e:
    print(f'⚠️  Could not load checkpoint: {e}')
    print('   Continuing with random weights — predictions will be uninformative.')


## 5. Fallback & Test File Detection

In [ ]:
# On Kaggle hidden evaluation, test_soundscapes/ exists and contains .ogg files.
# During notebook validation / dry-runs that folder is absent — we fall back
# to the first 5 training soundscapes so all downstream cells still execute.
TEST_DIR  = os.path.join(CFG.ROOT_DIR, 'test_soundscapes')
test_files = []
if os.path.exists(TEST_DIR):
    test_files = sorted(glob.glob(f'{TEST_DIR}/*.ogg'))

if len(test_files) == 0:
    print('⚠️  FALLBACK ACTIVE: test_soundscapes/ not found or empty.')
    print('    Using first 5 training soundscapes as a dry-run.')
    test_files = sorted(glob.glob(f'{CFG.SOUNDSCAPE_DIR}/*.ogg'))[:5]
    IS_DRY_RUN = True
else:
    print(f'✅ Found {len(test_files)} files in test_soundscapes/.')
    IS_DRY_RUN = False


## 6. Sliding Window Inference

In [ ]:
mel_transform   = T.MelSpectrogram(
    sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
    n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX
).to(device)
amplitude_to_db = T.AmplitudeToDB(top_db=80).to(device)

all_predictions, all_row_ids = [], []

print(f"\n{'='*20} Starting Inference {'='*20}")
for audio_path in tqdm(test_files, desc='Processing Soundscapes'):
    filename = os.path.basename(audio_path).replace('.ogg', '')
    try:
        y, _ = sf.read(audio_path, always_2d=True)
        y    = y.mean(axis=1)  # stereo → mono
    except Exception as e:
        print(f'Error reading {audio_path}: {e}')
        continue

    y_tensor       = torch.tensor(y, dtype=torch.float32).to(device)
    total_samples  = len(y_tensor)
    window_samples = CFG.SR * CFG.WINDOW_SECONDS
    n_segments     = math.ceil(total_samples / window_samples)

    for seg_idx in range(n_segments):
        start_sample = seg_idx * window_samples
        end_time_sec = (seg_idx + 1) * CFG.WINDOW_SECONDS
        row_id       = f'{filename}_{end_time_sec}'

        segment = y_tensor[start_sample: start_sample + window_samples]
        if len(segment) < window_samples:
            segment = F.pad(segment, (0, window_samples - len(segment)))

        with torch.no_grad():
            mel   = mel_transform(segment)
            mel   = amplitude_to_db(mel)
            mel   = (mel - mel.min()) / (mel.max() - mel.min() + 1e-6)
            image = torch.stack([mel, mel, mel]).unsqueeze(0)  # (1,3,F,T)
            probs = torch.sigmoid(model(image)).squeeze(0).cpu().numpy()

        all_row_ids.append(row_id)
        all_predictions.append(probs)


## 7. Submission Formatting

In [ ]:
print('\nFormatting submission...')
prediction_df = pd.DataFrame(all_predictions, columns=unique_labels)
submission_df = prediction_df.reindex(columns=submission_labels, fill_value=0.0)
submission_df.insert(0, 'row_id', all_row_ids)
submission_df = submission_df[['row_id'] + submission_labels]

# Sanity checks (mirrors pipeline 01)
expected_cols = len(submission_labels) + 1
if submission_df.shape[1] != expected_cols:
    raise ValueError(f'Submission has {submission_df.shape[1]} columns, expected {expected_cols}.')
if len(submission_df) != len(all_row_ids):
    raise ValueError(f'Submission has {len(submission_df)} rows, expected {len(all_row_ids)}.')
if submission_df.isnull().values.any():
    raise ValueError('Submission contains missing values.')

submission_df.to_csv('submission.csv', index=False)
print(f'✅ Submission saved → submission.csv  shape={submission_df.shape}')
display(submission_df.head(3))
